# 04. Maximum Likelihood Estimation (MLE)

**The Grand Mathematical Bridge: How statistical likelihood derives Mean Squared Error (MSE) and Cross-Entropy loss from first principles.**

---

## 1. What is Maximum Likelihood Estimation?

Suppose we observe a dataset of independent and identically distributed (i.i.d.) samples $\mathbf{X} = \{x_1, x_2, \dots, x_N\}$ generated from an underlying probability distribution parameterized by $\theta$.

The **Likelihood Function** $L(\theta; \mathbf{X})$ measures how probable it is that the observed data was generated by parameter $\theta$:

$$L(\theta; \mathbf{X}) = \prod_{i=1}^N p(x_i; \theta)$$

The **Maximum Likelihood Estimator (MLE)** chooses the parameter $\hat{\theta}_{MLE}$ that maximizes this likelihood:

$$\hat{\theta}_{MLE} = \arg\max_\theta \prod_{i=1}^N p(x_i; \theta)$$

### Why take the Logarithm? (The Log-Likelihood)
Multiplying thousands of small probabilities produces severe arithmetic underflow ($10^{-50} \to 0$). Taking the natural log converts products into sums:

$$\ell(\theta) = \log L(\theta; \mathbf{X}) = \sum_{i=1}^N \log p(x_i; \theta)$$

Since $\log$ is a strictly increasing monotonic function, $\arg\max L(\theta) \equiv \arg\max \ell(\theta)$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Visualizing Log-Likelihood of a Gaussian Dataset
np.random.seed(42)
true_mu, true_sigma = 5.0, 2.0
data = np.random.normal(true_mu, true_sigma, size=100)

mu_candidates = np.linspace(2.0, 8.0, 200)
log_likelihoods = []

for m in mu_candidates:
    # Log-likelihood under N(m, true_sigma^2)
    ll = np.sum(np.log(1.0 / (true_sigma * np.sqrt(2*np.pi))) - 0.5 * ((data - m)/true_sigma)**2)
    log_likelihoods.append(ll)

plt.figure(figsize=(8, 5))
plt.plot(mu_candidates, log_likelihoods, 'b-', linewidth=2)
plt.axvline(np.mean(data), color='r', linestyle='--', label=f'Sample Mean (MLE) = {np.mean(data):.3f}')
plt.title("Gaussian Log-Likelihood Function vs Candidate Mean $\mu$")
plt.xlabel("Candidate Mean $\mu$")
plt.ylabel("Log-Likelihood $\ell(\mu)$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


---

## 2. The Grand Bridge #1: Deriving Mean Squared Error (MSE) from MLE

In Linear Regression, assume the target $y$ is corrupted by additive zero-mean Gaussian noise:
$$y = \mathbf{w}^T \mathbf{x} + \epsilon \quad \text{where } \epsilon \sim \mathcal{N}(0, \sigma^2)$$

This means the conditional probability of $y$ given $\mathbf{x}$ is:
$$p(y|\mathbf{x}; \mathbf{w}) = \frac{1}{\sigma \sqrt{2\pi}} \exp\left( -\frac{(y - \mathbf{w}^T \mathbf{x})^2}{2\sigma^2} \right)$$

Let's write the total Log-Likelihood for $N$ training points:
$$\ell(\mathbf{w}) = \sum_{i=1}^N \left[ \log\left(\frac{1}{\sigma\sqrt{2\pi}}\right) - \frac{1}{2\sigma^2} (y_i - \mathbf{w}^T \mathbf{x}_i)^2 \right] = -N \log(\sigma\sqrt{2\pi}) - \frac{1}{2\sigma^2} \sum_{i=1}^N (y_i - \mathbf{w}^T \mathbf{x}_i)^2$$

To **maximize** $\ell(\mathbf{w})$, we drop the constants and **minimize the negative log-likelihood**:
$$\arg\max_{\mathbf{w}} \ell(\mathbf{w}) \iff \arg\min_{\mathbf{w}} \frac{1}{2\sigma^2} \sum_{i=1}^N (y_i - \mathbf{w}^T \mathbf{x}_i)^2 \iff \arg\min_{\mathbf{w}} \frac{1}{N} \sum_{i=1}^N (y_i - \mathbf{w}^T \mathbf{x}_i)^2$$

### Theorem:
**Minimizing Mean Squared Error (MSE) is mathematically identical to Maximum Likelihood Estimation under Gaussian noise!**


---

## 3. The Grand Bridge #2: Deriving Cross-Entropy from MLE

In Binary Classification, the target $y \in \{0, 1\}$ follows a Bernoulli distribution:
$$p(y|\mathbf{x}; \mathbf{w}) = \hat{y}^y (1 - \hat{y})^{1 - y} \quad \text{where } \hat{y} = \sigma(\mathbf{w}^T \mathbf{x})$$

The Log-Likelihood is:
$$\ell(\mathbf{w}) = \sum_{i=1}^N \log \left[ \hat{y}_i^{y_i} (1 - \hat{y}_i)^{1 - y_i} \right] = \sum_{i=1}^N \left[ y_i \log \hat{y}_i + (1 - y_i) \log (1 - \hat{y}_i) \right]$$

To **maximize** the Bernoulli log-likelihood, we **minimize the Negative Log-Likelihood (NLL)**:
$$\mathcal{L}_{NLL} = -\ell(\mathbf{w}) = -\sum_{i=1}^N \left[ y_i \log \hat{y}_i + (1 - y_i) \log (1 - \hat{y}_i) \right] \equiv \text{Binary Cross-Entropy!}$$

### Theorem:
**Minimizing Cross-Entropy loss is mathematically identical to Maximum Likelihood Estimation under a Categorical/Bernoulli distribution!**


In [ ]:
# Numerical MLE for Linear Regression vs OLS Formula
# Generate synthetic linear data y = 3*x + 2 + noise
X = np.random.uniform(-3, 3, size=(50, 1))
y = 3.0 * X[:, 0] + 2.0 + np.random.normal(0, 0.5, size=50)

# Negative Log Likelihood objective function to minimize
def nll_linear_regression(weights):
    w, b = weights
    y_pred = w * X[:, 0] + b
    # Gaussian NLL
    return 0.5 * np.sum((y - y_pred)**2)

res = minimize(nll_linear_regression, [0.0, 0.0])
w_mle, b_mle = res.x

print(f"MLE Fitted Parameters: Slope w = {w_mle:.4f}, Intercept b = {b_mle:.4f}")
print("MLE matches Ordinary Least Squares regression solution!")


---

## 4. Summary & Key Takeaways

1. **Maximum Likelihood Estimation (MLE)** finds parameters that maximize the probability of observing the training data.
2. We maximize **Log-Likelihood** $\ell(\theta) = \sum \log p(x_i; \theta)$ for numerical stability.
3. **Mean Squared Error (MSE)** is the exact MLE loss for Gaussian distributed errors.
4. **Binary / Categorical Cross-Entropy** is the exact MLE loss for Bernoulli / Categorical distributed labels.
